# 1 - İmport Data and Libraries

In [ ]:
%pip install xgboost lightgbm tensorflow catboost imblearn

In [ ]:
# Gerekli kütüphanelerin import edilmesi.
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from imblearn.under_sampling import NearMiss
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Veri setini İmport Edelim

In [ ]:
# Veri setinin GitHub platformundan çekilip projeye dahil edilmesi.
url = "https://raw.githubusercontent.com/oztrkahmet/Heart-Attack-Prediction/refs/heads/main/Data/heart_attack_china.csv"

In [ ]:
# Verisetinin değişkene tanımlanması.
data = pd.read_csv(url)

# 2 - Exploratory Data Analysis - EDA

In [ ]:
# Veri setinin yapısını görmek için basit bir önbilgi alalım.
data.head()

In [ ]:
# Kategorik sütunların belirlenmesi
categorical_columns = data.select_dtypes(include=['object']).columns.tolist()
print("\nKategorik sütunlar:", categorical_columns)

# Benzersiz değerlerin sayısını kontrol edelim
for col in categorical_columns:
    print(f"\n{col} sütunu benzersiz değerleri:")
    print(data[col].value_counts())

In [ ]:
# Veri setinin içeriğini tüm sütunları dahil olmak üzere ayrıntılı görmek için.
pd.set_option('display.max_columns', None)

In [ ]:
# Veri setinin ayrıntılı istatistiklerini ve özelliklerini görme.
print(data.describe(include='all'))

In [ ]:
# Tekrar eden satırların kontrolü:
tekrarlananlar = data.duplicated(keep="first")

In [ ]:
# Eğer var ise tekrarlanan indislerin görsel olarak çıktısı:
data[tekrarlananlar]

### Sütun İsimlendirme

In [ ]:
# Sütunların isimlerini değiştirelim:
yeni_isimler = {
    'Patient_ID': 'Hasta_ID',
    'Age': 'Yas',
    'Gender': 'Cinsiyet',
    'Smoking_Status': 'Sigara_Durumu',
    'Hypertension': 'Hipertansiyon',
    'Diabetes': 'Diyabet',
    'Obesity': 'Obezite',
    'Cholesterol_Level': 'Kolesterol_Seviyesi',
    'Air_Pollution_Exposure': 'Hava_Kirliligi_Maruziyeti',
    'Physical_Activity': 'Fiziksel_Aktivite',
    'Diet_Score': 'Diyet_Puani',
    'Stress_Level': 'Stres_Seviyesi',
    'Alcohol_Consumption': 'Alkol_Tuketimi',
    'Family_History_CVD': 'Aile_KVH_Gecmisi', # KVH: Kardiyovasküler Hastalık
    'Healthcare_Access': 'Saglik_Hizmetlerine_Erisim',
    'Rural_or_Urban': 'Kirsal_veya_Kentsel',
    'Region': 'Bolge',
    'Province': 'Eyalet',
    'Hospital_Availability': 'Hastane_Mevcudiyeti',
    'TCM_Use': 'GCT_Kullanimi', # GCT: Geleneksel Çin Tıbbı
    'Employment_Status': 'Istihdam_Durumu',
    'Education_Level': 'Egitim_Seviyesi',
    'Income_Level': 'Gelir_Seviyesi',
    'Blood_Pressure': 'Kan_Basinci',
    'Chronic_Kidney_Disease': 'Kronik_Bobrek_Hastaligi',
    'Previous_Heart_Attack': 'Onceki_Kalp_Krizi',
    'CVD_Risk_Score': 'KVH_Risk_Puani',
    'Heart_Attack': 'Kalp_Krizi' # Hedef değişken
}
data = data.rename(columns=yeni_isimler)

In [ ]:
print("DataFrame'in Yeni Sütun İsimleri:")
print(data.columns.tolist())

In [ ]:
# Veri setimizdeki Patient_ID sütunu işimize yaramayacağından ve gereksiz bir boyut oluşturacağından verisetinden çıkarıyoruz:
data = data.drop('Hasta_ID',axis=1)

### Sütunlar Hakkında Genel Bilgiler

In [ ]:
# Veri seti sütunları hakkında bilgiler edinme.
print("\n Veri Seti Bilgileri (Sütunlar, Veri Tipleri, Eksik Olmayan Değerler):")
data.info()

In [ ]:
# Sayısal sütunların istatistiksel özellikleri.
print("\n Sayısal Sütunlar İçin İstatistiksel Özet:")
print(data.describe().T) # .T ile çıktının transpozesini alarak daha iyi görünmesini sağladık.

In [ ]:
# Kategorik sütunların istatistiksel özellikleri.
print("\n Kategorik Sütunlar İçin İstatistiksel Özet:")
print(data.describe(include='object').T)

In [ ]:
# Her sütundaki benzersiz değer sayısını inceleyelim.
print("\n Her Sütundaki Benzersiz Değer Sayısı:")
print(data.nunique())

### Eksik Veri Kontrolü

In [ ]:
# Tüm veri setindeki eksik veri olan yerleri tespit edelim.
print(data.isnull().sum())

In [ ]:
missing_values = data.isnull().sum()

In [ ]:
missing_percentage = (missing_values / len(data)) * 100

In [ ]:
missing_info = pd.DataFrame({'Eksik Değer Sayısı': missing_values, 'Yüzde (%)': missing_percentage})

In [ ]:
missing_info = missing_info[missing_info['Eksik Değer Sayısı'] > 0].sort_values(by='Yüzde (%)', ascending=False)

In [ ]:
print("\n Eksik Verilerin Analizi:")
if missing_info.empty:
    print("Veri setinde eksik değer bulunmamakta.")
else:
    print(missing_info)

In [ ]:
# Eksik verileri grafik üzerinde görselleştirelim.
plt.figure(figsize=(12, 6))
sns.heatmap(data.isnull(), cbar=False, cmap='viridis')
plt.title('Eksik Verilerin Görselleştirilmesi')
plt.show()

In [ ]:
# Eksik veri bulunan sütundaki değerleri sayıları ile birlikte görelim.
print(data["Egitim_Seviyesi"].value_counts(dropna=False))

In [ ]:
# Eksik veri bulunan sütundaki değerleri kategori olarak görelim.
print(data["Egitim_Seviyesi"].unique())

In [ ]:
# Bu kısımda veri seti incelendiğinde "none" değerinin aslında eksik veri değil de okumamış kişi anlamında olduğu anlaşılmıştır.
# Bu sebeple modellerin "none" değerini eksik veri görme sorunundan kurtarmak için bu değeri "uneducated" olark değiştirdik
# ve veri setindeki değerleri bu yeni değerlere güncelledik.
data["Egitim_Seviyesi"] = data["Egitim_Seviyesi"].fillna("Uneducated")

In [ ]:
# veri güncellemesi sonrası kategorileri tekrar görüntüleyelim.
print(data["Egitim_Seviyesi"].unique())

In [ ]:
# verinin güncellenmesi sonrası eksik verinin varlığını tekrar kontrol edelim.
print(data.isnull().sum())

### Aykırı Değer Kontrolü

In [ ]:
# Sayısal değerlerde aykırı değerlerin olup olmadığını ve verilerin genel dağılımını görmek için yoğunluk, kutu ve histogram grafiklerinden yararlandık.
numerical_cols = ['Yas', 'Kan_Basinci', 'KVH_Risk_Puani']

for col in numerical_cols:
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    sns.histplot(data[col], kde=True)
    plt.title(f'{col} - Histogram')

    plt.subplot(1, 3, 2)
    sns.boxplot(y=data[col])
    plt.title(f'{col} - Box Plot')

    plt.subplot(1, 3, 3)
    sns.kdeplot(data[col])
    plt.title(f'{col} - KDE Plot')

    plt.tight_layout()
    plt.show()

    print(f"\n{col} İstatistikleri:")
    print(data[col].describe())

### Hedef ve diğer niteliklerin Görselleştirilmesi

In [ ]:
# Hedef değişkenin dağılımını grafiksel olarak görüntüleyelim.
plt.figure(figsize=(6, 4))
sns.countplot(x='Kalp_Krizi', data=data)
plt.title('Kalp Krizi (Hedef Değişken) Dağılımı')
plt.xlabel('Kalp Krizi Geçirme Durumu')
plt.ylabel('Hasta Sayısı')
plt.show()

# Oranları da yazdırabilirsin
print("Kalp Krizi Yüzdesel Sınıf Dağılımı:")
print(data['Kalp_Krizi'].value_counts(normalize=True) * 100)

In [ ]:
# Object türündeki sütunlarda bulunan kategorilerin baskınlık durumlarını görselleştirerek veriyi inceledik.
# Hedef değişkeni almadık.
categorical_cols = data.select_dtypes(include='object').columns.tolist()
# Eğer hedef değişken object ise görselleştirmeden çıkarılsın olarak belirledik.
if 'Kalp_Krizi' in categorical_cols: categorical_cols.remove('Kalp_Krizi')

# Kullanılacak renk paletini seçelim
palette_choice = 'tab10'

for col in categorical_cols:
    plt.figure(figsize=(10, 5))
    # kategori sayısı çok fazlaysa y ekseninde gösterilmesi için
    if data[col].nunique() > 5:
        sns.countplot(y=col, data=data, order = data[col].value_counts().index, palette=palette_choice, hue=col, legend=False)
        plt.title(f'{col} Dağılımı (Frekans)')
        plt.xlabel('Hasta Sayısı')
        plt.ylabel(col)
    else:
        sns.countplot(x=col, data=data, order = data[col].value_counts().index, palette=palette_choice, hue=col, legend=False)
        plt.title(f'{col} Dağılımı (Frekans)')
        plt.xlabel(col)
        plt.ylabel('Hasta Sayısı')
    plt.show()
    print(f"\n{col} Değer Sayıları:")
    print(data[col].value_counts())
    print("-" * 30)

In [ ]:
# Sayısal verilerin hedef değişken ile karşılaştırılması ve kalp krizi geçiren ve geçirmeyen gruplar arasında nasıl farklılaştığını inceleyelim.
for col in numerical_cols:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x='Kalp_Krizi', y=col, data=data)
    plt.title(f'{col} vs Kalp Krizi')
    plt.xlabel('Kalp Krizi Geçirme Durumu (No/Yes)')
    plt.ylabel(col)
    plt.show()

    # Histogramları yan yana görelim.
    sns.displot(data=data, x=col, col='Kalp_Krizi', kde=True)
    plt.suptitle(f'{col} Dağılımı - Kalp Krizi Gruplarına Göre', y=1.02)
    plt.show()

In [ ]:
# Kategorik özelliklerin kalp krizi riski ile ilişkisini incelemek istersek:
for col in categorical_cols:
    plt.figure(figsize=(10, 6))
    # Yüzdelik oranlar için crosstab kullanalım
    ct = pd.crosstab(data[col], data['Kalp_Krizi'], normalize='index') # 'index'e göre normalize et
    ct.plot(kind='bar', stacked=True, figsize=(10,6))
    plt.title(f'{col} Kategorilerine Göre Kalp Krizi Oranları')
    plt.xlabel(col)
    plt.ylabel('Oran')
    plt.legend(title='Kalp_Krizi')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Sayısal olarak değerlerini görelim:
    print(f"\n{col} vs Kalp Krizi Oranı:")
    print(pd.crosstab(data[col], data['Kalp_Krizi']))
    print("-" * 30)

In [ ]:
# Korelasyon Matrisi
plt.figure(figsize=(8, 6))
correlation_matrix = data[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".3f")
plt.title('Sayısal Değişkenler Arası Korelasyon')
plt.show()

print("\nKorelasyon Matrisi:")
print(correlation_matrix)

In [ ]:
# Pair Plot ile sayısal değişkenler arası ilişkiye bakalım:
sns.pairplot(data[numerical_cols + ['Kalp_Krizi']], hue='Kalp_Krizi')
plt.suptitle('Sayısal Değişkenler Arası İlişki (Kalp Krizi Rengine Göre)', y=1.02)
plt.show()

# 3 - Nitelik Dönüşümleri ve Verisetini Ayırma

In [ ]:
# Kategorik ve sayısal sütunları ayır
kategorik_sutunlar = ['Cinsiyet','Sigara_Durumu','Hipertansiyon','Diyabet','Obezite','Kolesterol_Seviyesi',
    'Hava_Kirliligi_Maruziyeti','Fiziksel_Aktivite','Diyet_Puani','Stres_Seviyesi','Alkol_Tuketimi','Aile_KVH_Gecmisi',
    'Saglik_Hizmetlerine_Erisim','Kirsal_veya_Kentsel','Bolge','Eyalet','Hastane_Mevcudiyeti','GCT_Kullanimi',
    'Istihdam_Durumu','Egitim_Seviyesi','Gelir_Seviyesi','Kronik_Bobrek_Hastaligi','Onceki_Kalp_Krizi']

sayisal_sutunlar = ['Yas', 'Kan_Basinci', 'KVH_Risk_Puani']

### Sütunlara Label Encoder Uygulanması

In [ ]:
# Kategorik özelliklere Label Encoder uygulayalım
le_features = LabelEncoder()
for sutun in kategorik_sutunlar:
    data[sutun] = le_features.fit_transform(data[sutun])

In [ ]:
# Hedef değişkene de ('Heart_Attack') Label Encoder uygulayalım
le_target = LabelEncoder()
data['Kalp_Krizi'] = le_target.fit_transform(data['Kalp_Krizi'])

### Sayısal Niteliklerin Ölçeklenmesi

In [ ]:
# Sayısal sütunlara Standard Scaler uygulayalım
scaler = StandardScaler()
data[sayisal_sutunlar] = scaler.fit_transform(data[sayisal_sutunlar])

### Hedef ve Özellik Sütunlarını Ayıralım

In [ ]:
# Hedef değişkeni ve özellikleri ayırdık
X = data.drop('Kalp_Krizi', axis=1)
y = data['Kalp_Krizi']

### Near Miss Undersampler ile Hedef Sınıf Dengesizliği Giderme

In [ ]:
# Sınıf dengesizliğini kontrol ettik
print("Sınıf dağılımı (önce):")
print(y.value_counts())


# NearMiss Version 2 ile hedeh sınıfı dengesizliğini giderelim
nm = NearMiss(version=2, n_jobs=-1)
X_resampled, y_resampled = nm.fit_resample(X, y)

In [ ]:
# Dengeleme sonrası kontrol edelim
print("\nSınıf dağılımı (sonra):")
print(pd.Series(y_resampled).value_counts())

# 4 - Modellerin Eğitilmesi

In [ ]:
# Makine öğrenmesi modellerini tanımladık
models = {
    'Logistic Regression': LogisticRegression(random_state=12, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=12),
    'Random Forest': RandomForestClassifier(random_state=12, n_estimators=100, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=12, n_estimators=100),
    'AdaBoost': AdaBoostClassifier(random_state=12, n_estimators=100),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes': GaussianNB(),
    'Neural Network (MLP)': MLPClassifier(random_state=12, max_iter=1000, early_stopping=True),
    'XGBoost': XGBClassifier(random_state=12, eval_metric='logloss', n_jobs=-1),
    'LightGBM': LGBMClassifier(random_state=12, verbose=-1, n_jobs=-1),
    'CatBoost': CatBoostClassifier(random_state=12, verbose=False)
}

# Sonuçları saklamak için sözlük oluşturalım
results = {}
confusion_matrices = {}

### Çapraz Doğrulama ile Model Eğitimi

In [ ]:
# Çapraz doğrulama için StratifiedKFold tanımladık
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)

print("\n\nMakine Öğrenmesi Modelleri 5-Katlı Çapraz Doğrulama ile Değerlendiriliyor...")
# Her model için eğitim ve değerlendirme yapıyoruz
for name, model in models.items():
    print(f"{name} değerlendiriliyor...")

    # Metrikleri tanımla
    scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

    # cross_validate ile model performansını birden çok metrikle ölç
    cv_results = cross_validate(
        estimator=model,
        X=X_resampled,
        y=y_resampled,
        cv=cv_strategy,
        scoring=scoring_metrics,
        n_jobs=-1
    )

    # Metriklerin ortalamasını al ve sonuçlara kaydet
    results[name] = {
        'Accuracy': cv_results['test_accuracy'].mean(),
        'Precision': cv_results['test_precision'].mean(),
        'Recall': cv_results['test_recall'].mean(),
        'F1-Score': cv_results['test_f1'].mean(),
        'ROC-AUC': cv_results['test_roc_auc'].mean()
    }

    # Confusion matrix için çapraz doğrulama tahminleri al
    y_pred = cross_val_predict(
        estimator=model,
        X=X_resampled,
        y=y_resampled,
        cv=cv_strategy,
        n_jobs=-1
    )

    # Confusion matrix sakla
    confusion_matrices[name] = confusion_matrix(y_resampled, y_pred)

# 5 - Derin Öğrenme Model Eğitimi

### Çapraz Doğrulama için Derin Öğrenme Modelinin Tanımlanması

In [ ]:
# --- DERİN ÖĞRENME MODELİ İÇİN MANUEL ÇAPRAZ DOĞRULAMA ---
print("\n\nDerin Öğrenme Modeli 5-Katlı Çapraz Doğrulama ile Değerlendiriliyor...")

# Her katlamada (fold) yeniden oluşturulabilmesi için model oluşturma fonksiyonu tanımladık
def create_dl_model(input_shape):
    model = keras.Sequential([
        keras.Input(shape=(input_shape,)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(32, activation='relu'),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    return model


In [ ]:
# Early stopping in tanımlanması
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True, verbose=0
)

In [ ]:
# Metrikleri ve tahminleri saklamak için listeler tanımladık
dl_scores = {'Accuracy': [], 'Precision': [], 'Recall': [], 'F1-Score': [], 'ROC-AUC': []}
all_y_true = []
all_y_pred_dl = []

In [ ]:
# X ve y'yi numpy array'e çevirerek daha hızlı indeksleme sağlayabiliriz
X_np = X_resampled.to_numpy()
y_np = y_resampled.to_numpy()

In [ ]:
for fold, (train_index, val_index) in enumerate(cv_strategy.split(X_np, y_np)):
    print(f"Derin Öğrenme - Fold {fold+1}/5")
    X_train, X_val = X_np[train_index], X_np[val_index]
    y_train, y_val = y_np[train_index], y_np[val_index]

    # Her fold için modeli yeniden oluştur (ağırlıkların sızmasını önle)
    dl_model = create_dl_model(X_train.shape[1])

    # Modeli eğit
    dl_model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_data=(X_val, y_val), # Katmanın validasyon setini kullan
        callbacks=[early_stopping],
        verbose=0
    )

    # Tahminler
    y_pred_proba_dl = dl_model.predict(X_val).flatten()
    y_pred_dl = (y_pred_proba_dl > 0.5).astype(int)

    # Metrikleri hesapla ve listelere ekle
    dl_scores['Accuracy'].append(accuracy_score(y_val, y_pred_dl))
    dl_scores['Precision'].append(precision_score(y_val, y_pred_dl, zero_division=0))
    dl_scores['Recall'].append(recall_score(y_val, y_pred_dl, zero_division=0))
    dl_scores['F1-Score'].append(f1_score(y_val, y_pred_dl, zero_division=0))
    dl_scores['ROC-AUC'].append(roc_auc_score(y_val, y_pred_proba_dl))

    # Confusion matrix için tüm tahminleri ve gerçek değerleri topla
    all_y_true.extend(y_val)
    all_y_pred_dl.extend(y_pred_dl)

In [ ]:
# Ortalama metrikleri hesapla
dl_results = {metric: np.mean(values) for metric, values in dl_scores.items()}

# Derin öğrenme sonuçlarını ekle
results['Deep Learning'] = dl_results
confusion_matrices['Deep Learning'] = confusion_matrix(all_y_true, all_y_pred_dl)

print(f"\nDerin Öğrenme Ortalama Performansı:")
for metric, value in dl_results.items():
    print(f"{metric}: {value:.4f}")

### DL ve ML Modelleri Değerlendirme

In [ ]:
# Tüm sonuçları DataFrame olarak göster (DL dahil)
results_df_all = pd.DataFrame(results).T
print("\n\nTÜM MODELLERİN 5-KATLI ÇAPRAZ DOĞRULAMA PERFORMANS KARŞILAŞTIRMASI:")
print("="*80)
print(results_df_all.sort_values(by='F1-Score', ascending=False).round(4))
print("="*80)

# 6 - Sonuçlar ve Görsel Grafikler

### Karar Matrisi ile Tüm Modellerin Karşılaştırılması

In [ ]:
# Model sayısına göre subplot düzenini ayarla
num_total_models_for_cm = len(confusion_matrices)
cols_cm = 4
rows_cm = (num_total_models_for_cm + cols_cm -1) // cols_cm

if num_total_models_for_cm > 0:
    fig_cm, axes_cm = plt.subplots(rows_cm, cols_cm, figsize=(5*cols_cm, 4*rows_cm))
    axes_cm = axes_cm.ravel()

    for idx, (name, cm) in enumerate(confusion_matrices.items()):
        if idx < len(axes_cm):
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_cm[idx])
            axes_cm[idx].set_title(f'{name}')
            axes_cm[idx].set_xlabel('Predicted')
            axes_cm[idx].set_ylabel('Actual')

    for i in range(num_total_models_for_cm, len(axes_cm)):
        fig_cm.delaxes(axes_cm[i])

    plt.tight_layout()
    plt.suptitle('Tüm Modellerin Toplam Confusion Matrix Sonuçları (5-Katlı CV)', fontsize=16, y=1.02 if rows_cm > 1 else 1.05)
    plt.show()

### Tüm Modellerin Performans Karşılaştırma Grafiği

In [ ]:
# Tüm Modellerin (ML + DL) Performans karşılaştırma grafiği
if not results_df_all.empty:
    fig_perf, ax_perf = plt.subplots(figsize=(14, 8))
    metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
    num_metrics = len(metrics_to_plot)
    num_total_models_for_perf_chart = len(results_df_all)

    total_bar_width_factor = 0.8
    bar_width = total_bar_width_factor / num_metrics # Bar genişliğini metrik sayısına göre ayarla

    x_indices = np.arange(len(results_df_all))

    for i, metric in enumerate(metrics_to_plot):
        values = results_df_all[metric]
        bar_positions = x_indices - (total_bar_width_factor / 2) + (i * bar_width) + (bar_width / 2)
        ax_perf.bar(bar_positions, values, bar_width, label=metric)

    ax_perf.set_xlabel('Modeller', fontsize=12)
    ax_perf.set_ylabel('Ortalama Skor (5-Katlı CV)', fontsize=12)
    ax_perf.set_title('Tüm Modellerin Performans Karşılaştırması (ML ve DL)', fontsize=15)
    ax_perf.set_xticks(x_indices)
    ax_perf.set_xticklabels(results_df_all.index, rotation=45, ha="right")
    ax_perf.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Metrikler")
    ax_perf.grid(True, linestyle='--', alpha=0.7)
    ax_perf.set_ylim(0, 1.05)

    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()